In [6]:
# Устанавливаем специальный пакет для работы Selenium в Colab
!pip install selenium google-colab-selenium pandas bs4 -q
print("✅ Библиотеки установлены!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.0 MB/s eta 0:00:00
✅ Библиотеки установлены!


In [7]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from google.colab import files
import google_colab_selenium as gs # Импортируем нашу спасительную библиотеку

print("1. Запускаем виртуальный браузер (безопасный режим)...")
# gs.Chrome() автоматически настраивает драйвер, чтобы он не падал в Colab
driver = gs.Chrome()

url = "https://www.rikicollection.com/catalog"
driver.get(url)
print("2. Сайт открыт. Начинаем прокрутку вниз для подгрузки товаров...")

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_attempts = 0

# Цикл прокрутки страницы
while True:
    # Крутим страницу в самый низ
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3) # Ждем 3 секунды, пока Тильда подгрузит новые товары

    # Пробуем найти кнопку "Показать еще" (если она есть) и нажать её
    try:
        btn = driver.find_element(By.CSS_SELECTOR, '.t-store__load-more-btn')
        if btn.is_displayed():
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
    except:
        pass # Кнопки нет — работает автоподгрузка по скроллу

    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        scroll_attempts += 1
        # Если страница не удлиняется 3 проверки подряд — значит, дно достигнуто
        if scroll_attempts >= 3:
            print("3. Конец страницы достигнут. Все товары на экране!")
            break
    else:
        scroll_attempts = 0
        last_height = new_height

# Забираем HTML-код, в котором теперь отрендерены вообще все товары
html = driver.page_source
driver.quit()

print("4. Извлекаем данные из карточек...")

soup = BeautifulSoup(html, 'html.parser')
products = soup.find_all('div', class_='js-product')

print(f"Найдено товаров: {len(products)}")

all_products = []

for product in products:
    name_tag = product.find(class_='js-product-name')
    name = name_tag.text.strip() if name_tag else "Не указано"

    sku_tag = product.find(class_='js-product-sku')
    sku = sku_tag.text.strip() if sku_tag else "Не указан"

    descr_tag = product.find(class_='js-store-prod-descr')
    descr = descr_tag.text.strip() if descr_tag else ""

    price_tag = product.find(class_='js-product-price')
    price = price_tag.text.strip().replace(" ", "") if price_tag else "0"

    product_url = product.get('data-product-url', '')
    if not product_url and product.find('a'):
        product_url = product.find('a').get('href', '')

    img_tag = product.find(class_='js-product-img')
    img_url = img_tag.get('data-original', '') if img_tag else ""

    options_tags = product.select('.js-product-edition-option-variants option')
    materials = [opt.text.strip() for opt in options_tags] if options_tags else []

    item_info = {
        "Название": name,
        "Артикул": sku,
        "Категория": descr,
        "Цена (руб)": price,
        "Материалы": ", ".join(materials),
        "Ссылка на товар": product_url,
        "Ссылка на фото": img_url
    }

    if item_info not in all_products:
        all_products.append(item_info)

print(f"\nСбор завершен! Уникальных товаров собрано: {len(all_products)}")

if all_products:
    df = pd.DataFrame(all_products)
    display(df.head())

    filename = "riki_selenium_fixed.xlsx"
    df.to_excel(filename, index=False)
    print(f"\nСохраняем файл {filename}...")
    files.download(filename)

1. Запускаем виртуальный браузер (безопасный режим)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2. Сайт открыт. Начинаем прокрутку вниз для подгрузки товаров...
3. Конец страницы достигнут. Все товары на экране!
4. Извлекаем данные из карточек...
Найдено товаров: 62

Сбор завершен! Уникальных товаров собрано: 62


,Название,Артикул,Категория,Цена (руб),Материалы,Ссылка на товар,Ссылка на фото
0,Крош,SMSH.1.01,ЗАПОНКИ,15000,"Серебро 925, Белое золото 585, Желтое золото 7...",https://www.rikicollection.com/tproduct/416976...,https://static.tildacdn.com/tild3064-6137-4532...
1,Крош,SMSH.2.01,ПОДВЕСКИ,8900,"Серебро 925, Белое золото 585, Белое золото 75...",https://www.rikicollection.com/tproduct/416976...,https://static.tildacdn.com/tild6334-6134-4664...
2,Крош,SMSH.1.01,"Запонка, Серебро 925",15000,Серебро 925,https://www.rikicollection.com/catalog-krosh/t...,https://static.tildacdn.com/tild3064-6137-4532...
3,Крош,SMSH.3.01,СЕРЕЖКА - ГВОЗДИК,4500,"Серебро 925, Белое золото 585, Белое золото 75...",https://www.rikicollection.com/catalog-krosh/t...,https://static.tildacdn.com/tild6138-3638-4639...
4,Крош,SMSH.4.01,ТОНКИЙ БРАСЛЕТ,6950,"Серебро 925, Белое золото 585, Белое золото 75...",https://www.rikicollection.com/catalog-krosh/t...,https://static.tildacdn.com/tild3066-6238-4664...



Сохраняем файл riki_selenium_fixed.xlsx...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import time
import pandas as pd
from bs4 import BeautifulSoup
import google_colab_selenium as gs
from google.colab import files

# Запуск браузера
driver = gs.Chrome()
url = "https://dvamyacha.com/landing/Smeshariki"
driver.get(url)
time.sleep(5)

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, 'html.parser')
products = soup.find_all('div', class_='catalog-list-item')

# Разметка персонажей
mapping = {
    "99": "Крош",
    "z2": "Пин",
    "g4": "Лосяш",
    "x1": "Нюша"
}

unique_products = {}

for item in products:
    name_tag = item.find('a', class_='catalog-list-item-name')
    if not name_tag: continue

    link = "https://dvamyacha.com" + name_tag['href']
    if link in unique_products: continue

    # Очистка цены
    price_tag = item.find('div', class_='catalog-list-item-price')
    clean_price = price_tag.get_text(separator='|', strip=True).split('|')[0].strip() if price_tag else "0"

    # Фото
    img_tag = item.find('img', class_='catalog-list-item-img')
    img_url = "https://dvamyacha.com" + img_tag['src'] if img_tag else ""

    # Определяем персонажа
    char_name = "Неизвестно"
    for key, name in mapping.items():
        if key in link:
            char_name = name
            break

    # ФОРМИРОВАНИЕ ИМЕНИ ТОВАРА
    # Вместо name_tag.text.strip() принудительно задаем формат:
    item_title = f"Кеды с {char_name}" if char_name != "Неизвестно" else name_tag.text.strip()

    unique_products[link] = {
        "Персонаж": char_name,
        "Товар": item_title,  # Теперь здесь всегда "Кеды с ..."
        "Цена": clean_price,
        "Ссылка": link,
        "Фото": img_url
    }

# Создаем таблицу
df = pd.DataFrame(list(unique_products.values()))

# Сортируем
order = ["Крош", "Пин", "Лосяш", "Нюша"]
df['Персонаж'] = pd.Categorical(df['Персонаж'], categories=order, ordered=True)
df = df.sort_values('Персонаж')

display(df)
df.to_excel("smeshariki_final.xlsx", index=False)
files.download("smeshariki_final.xlsx")

<IPython.core.display.Javascript object>

,Персонаж,Товар,Цена,Ссылка,Фото
3,Крош,Кеды с Крош,7 900 ₽,https://dvamyacha.com/catalog/kedy/kedy_dba_x_...,https://dvamyacha.com/upload/iblock/319/6p3tlc...
2,Пин,Кеды с Пин,7 900 ₽,https://dvamyacha.com/catalog/kedy/kedy_dba_x_...,https://dvamyacha.com/upload/iblock/5b5/l9eea4...
1,Лосяш,Кеды с Лосяш,7 900 ₽,https://dvamyacha.com/catalog/kedy/kedy_dba_x_...,https://dvamyacha.com/upload/iblock/5ad/yv62wl...
0,Нюша,Кеды с Нюша,7 900 ₽,https://dvamyacha.com/catalog/kedy/kedy_dba_x_...,https://dvamyacha.com/upload/iblock/a5f/5giu7u...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import time
import pandas as pd
from bs4 import BeautifulSoup
import google_colab_selenium as gs
from google.colab import files
from selenium.webdriver.common.by import By

# 1. Запуск
driver = gs.Chrome()
url = "https://smlerch.ru/#shop"
driver.get(url)
time.sleep(3)

# 2. Прожимаем кнопку «Загрузить ещё» (как делали раньше)
while True:
    try:
        btn = driver.find_element(By.CLASS_NAME, "js-store-load-more-btn")
        if btn.is_displayed():
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(3)
        else:
            break
    except:
        break

# 3. Собираем ссылки на товары со Смешариками
soup = BeautifulSoup(driver.page_source, 'html.parser')
product_links = []
for item in soup.select('.js-product'):
    name_tag = item.select_one('.js-product-name')
    if name_tag and "смешарики" in name_tag.text.lower():
        link = item.find('a')['href']
        product_links.append(link)

print(f"Найдено товаров: {len(product_links)}. Начинаю сбор данных...")

# 4. Заходим в каждый товар
all_data = []
for link in product_links:
    driver.get(link)
    time.sleep(2) # Даем время прогрузиться описанию

    prod_soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Собираем данные
    name = prod_soup.select_one('.js-product-name').text.strip() if prod_soup.select_one('.js-product-name') else ""
    price = prod_soup.select_one('.js-store-prod-price-val').text.strip() if prod_soup.select_one('.js-store-prod-price-val') else "0"

    # Теперь берем полное описание (оно лежит в блоке .t-store__prod-popup__text или похожем)
    descr_tag = prod_soup.select_one('.t-store__prod-popup__text')
    description = descr_tag.text.strip() if descr_tag else "Нет описания"

    all_data.append({
        "Название": name,
        "Цена": price,
        "Описание": description,
        "Ссылка": link
    })
    print(f"Собрал: {name}")

driver.quit()

# 5. Сохранение
df = pd.DataFrame(all_data)
df.to_excel("smlerch_full_data.xlsx", index=False)
files.download("smlerch_full_data.xlsx")
print("Готово! Данные сохранены.")

<IPython.core.display.Javascript object>

Найдено товаров: 2. Начинаю сбор данных...
Собрал: Футболка "Смешарики + Лубок" белая
Собрал: Футболка "Смешарики + Лубок" чёрная


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Готово! Данные сохранены.
